# Multi-scale Poisson (Laplacian)

Steady problem $-\Delta u = f$ on $[0,1]^2$ whose exact solution superposes
several spatial frequencies, $u=\tfrac1n\sum_w \sin(w\pi x)\sin(w\pi y)$ with $w\in\{4,8,12,16\}$.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import jax
import jax.numpy as jnp

import pinn
from pinn import operators as op, sampling

%matplotlib inline
jax.config.update("jax_enable_x64", True)   # double precision

In [ ]:
class LaplacianProblem(pinn.Problem):
    """-Delta u = f on [0, 1]^2, multi-scale exact solution (steady)."""

    x_min, x_max = 0.0, 1.0
    ws = (4, 8, 12, 16)
    problem_name = "Laplacian"
    ref_path = pinn.reference_path("laplacian")

    def __init__(self, *, n_pde, n_bc, n_ic=0):
        self.n_pde, self.n_bc, self.n_ic = n_pde, n_bc, n_ic

    def residual_fns(self):
        return {"pde": self.pde_residual, "bc": self.bc_residual}

    def pde_residual(self, model, coords):
        u = lambda c: model(c)[0]
        lap = op.laplacian(u, coords, (0, 1))
        f = (2.0 / len(self.ws)) * sum(
            (w * jnp.pi)**2 * jnp.sin(w * jnp.pi * coords[0]) * jnp.sin(w * jnp.pi * coords[1])
            for w in self.ws)
        return jnp.array([-lap - f])

    def bc_residual(self, model, coords):
        return jnp.array([model(coords)[0]])

    def samplers(self):
        return {"pde": sampling.box(self.x_min, self.x_max, 2),
                "bc":  sampling.boundary_faces(self.x_min, self.x_max, 2)}

In [ ]:
cfg = pinn.RunConfig(
    network=lambda key: pinn.SIREN(
        key, LaplacianProblem, time_dependent=False, n_fourier=4,
        hidden_dims=(60, 60, 60, 60), periodic_bc=False, n_inputs=2, n_outputs=1,
    ),
    n_pde=2**15, n_bc=2**14,
    residual_sketch=4000, parameter_sketch=4000,
    batch_size=2**14, probe_batch_size=2**10,
    pde_weight=1e-8, bc_weight=1.0,
)

In [ ]:
pinn.precompile64(LaplacianProblem, cfg)

In [ ]:
results = pinn.train64(LaplacianProblem, cfg)

## Results

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ref = pinn.load_reference("laplacian")
x0, x1 = results["plot_x0"], results["plot_x1"]
lx, ly = results["plot_axes"]
extent = [x0[0], x0[-1], x1[0], x1[-1]]

for ch in ref.channels:
    pred = np.array(results["u_pred_plot"][ch])
    exact = np.array(ref.plot_grids[ch])
    err = np.abs(pred - exact)
    rel = np.linalg.norm(pred - exact) / np.linalg.norm(exact)

    fig, axs = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
    for ax, data, title, cmap in zip(
        axs, [pred, exact, err],
        [f"PINN  ${ch}$", f"reference  ${ch}$", f"abs error  (rel $\\ell_2$={rel:.2e})"],
        ["RdBu_r", "RdBu_r", "magma"],
    ):
        im = ax.imshow(data.T, origin="lower", aspect="auto", extent=extent, cmap=cmap)
        ax.set(xlabel=f"${lx}$", ylabel=f"${ly}$", title=title)
        fig.colorbar(im, ax=ax)
    plt.show()